# YOLO26 starter: detecting partially occluded objects

Run the cells from top to bottom. Before you start:

1. Use **Runtime > Change runtime type** and pick a GPU (for example T4).
2. Zip your dataset as `occlusionObjects.zip` and upload it to the top level of Google Drive.
3. Edit the **Configuration** cell so the names and classes match your project.

Expected dataset layout inside the zip:

```
occlusionObjects/
  images/train   images/val   images/test/clear   images/test/half   images/test/mostly
  labels/train   labels/val   labels/test/clear   labels/test/half   labels/test/mostly
```

The `images/test/...` folders are optional and only used by the occlusion-level evaluation near the end.

## 1. Install and check the GPU

In [ ]:
# Install the Ultralytics package that provides YOLO26
!pip install -q -U ultralytics
# Show the attached GPU; if this prints an error then change the runtime type to a GPU
!nvidia-smi

## 2. Configuration

In [ ]:
from pathlib import Path
# Name of the zip file that you uploaded to the top level of your Google Drive
datasetZipName = "occlusionObjects.zip"
# Name of the folder that appears after unzipping, which must match the folder inside the zip
datasetFolderName = "occlusionObjects"
# Class names in the same order as the class numbers in your label files
classNames = ["mug", "bottle", "apple"]
# Pretrained model to start from; n is nano, then s, m, l and x for bigger models
baseModelName = "yolo26n.pt"
# Number of full passes over the training data
trainingEpochs = 100
# Image size in pixels that every image is resized to during training
imageSize = 640
# Occlusion levels used by the optional evaluation near the end
occlusionLevels = ["clear", "half", "mostly"]
# Drive folder for results, so that weights survive when Colab resets the runtime
driveResultsFolder = Path("/content/drive/MyDrive/yoloProjectResults")
# Local copy of the zip file on the Colab machine
localZipPath = Path("/content") / datasetZipName
# Local folder that will hold the unzipped dataset
datasetRootPath = Path("/content/datasets") / datasetFolderName

## 3. Mount Google Drive and unpack the dataset

The zip is copied to local storage first because training directly from Drive is slow.

In [ ]:
from google.colab import drive
# Mount Google Drive; a pop-up will ask for permission
drive.mount("/content/drive")
# Make sure the results folder exists
driveResultsFolder.mkdir(parents=True, exist_ok=True)
# Copy the zip to local storage because training straight from Drive is slow
!cp "/content/drive/MyDrive/{datasetZipName}" "{localZipPath}"
# Unzip quietly into the datasets folder and overwrite any older copy
!unzip -q -o "{localZipPath}" -d /content/datasets

## 4. Helper functions

In [ ]:
from IPython.display import display
from PIL import Image
from ultralytics import YOLO
# File endings that count as images
imageSuffixes = {".jpg", ".jpeg", ".png"}

# Write a dataset.yaml file so that YOLO knows where the images are and what the classes are called
# The root folder must contain images/train, images/val, labels/train and labels/val
def writeDatasetYaml(rootPath, classNames, yamlName="dataset.yaml"):
    yamlPath = rootPath / yamlName
    lines = [f"path: {rootPath}", "train: images/train", "val: images/val", "names:"]
    for classNumber, className in enumerate(classNames):
        lines.append(f"  {classNumber}: {className}")
    yamlPath.write_text("\n".join(lines) + "\n")
    return yamlPath

# Count the images and label files of one split and warn about empty folders
# A missing label file is allowed because an image with no objects needs no label file
def checkSplit(rootPath, splitName):
    imageFolder = rootPath / "images" / splitName
    labelFolder = rootPath / "labels" / splitName
    imageFiles = [file for file in imageFolder.glob("*") if file.suffix.lower() in imageSuffixes]
    labelFiles = list(labelFolder.glob("*.txt"))
    print(f"{splitName}: {len(imageFiles)} images, {len(labelFiles)} label files")
    if len(imageFiles) == 0: print(f"  WARNING: no images found in {imageFolder}")
    if len(labelFiles) == 0: print(f"  WARNING: no labels found in {labelFolder}")

# Check that every class number in the label files exists in the class list
# An out-of-range number usually means the classes were ordered differently during annotation
def checkClassNumbers(rootPath, splitName, classCount):
    labelFolder = rootPath / "labels" / splitName
    for labelFile in labelFolder.glob("*.txt"):
        for line in labelFile.read_text().splitlines():
            if not line.strip(): continue
            classNumber = int(line.split()[0])
            if classNumber >= classCount: print(f"  WARNING: class {classNumber} in {labelFile.name} is out of range")

# Display the first few annotated images from a folder inside the notebook
def showSavedImages(folderPath, maximumCount=4):
    imageFiles = [file for file in sorted(Path(folderPath).iterdir()) if file.suffix.lower() in imageSuffixes]
    for imageFile in imageFiles[:maximumCount]:
        display(Image.open(imageFile))

# Evaluate the model separately for each occlusion level
# Expected layout is images/test/<level> with matching labels/test/<level> folders
# Comparing the scores shows how much accuracy drops as objects become more hidden
def evaluateByOcclusionLevel(model, rootPath, classNames, levelNames):
    scoresByLevel = {}
    for levelName in levelNames:
        levelFolder = rootPath / "images" / "test" / levelName
        if not levelFolder.exists(): print(f"Skipping {levelName}: {levelFolder} does not exist"); continue
        nameLines = [f"  {classNumber}: {className}" for classNumber, className in enumerate(classNames)]
        yamlLines = [f"path: {rootPath}", f"train: images/test/{levelName}", f"val: images/test/{levelName}", "names:"] + nameLines
        levelYamlPath = rootPath / f"dataset_{levelName}.yaml"
        levelYamlPath.write_text("\n".join(yamlLines) + "\n")
        levelMetrics = model.val(data=str(levelYamlPath), verbose=False, plots=False)
        scoresByLevel[levelName] = levelMetrics.box.map50
    return scoresByLevel

## 5. Create dataset.yaml and sanity-check the data

Read any warnings carefully. Most training problems are really data problems.

In [ ]:
# Generate the dataset.yaml file from the class list in the configuration cell
datasetYamlPath = writeDatasetYaml(datasetRootPath, classNames)
for splitName in ["train", "val"]:
    checkSplit(datasetRootPath, splitName)
    checkClassNumbers(datasetRootPath, splitName, len(classNames))
print(datasetYamlPath.read_text())

## 6. Optional baseline: what does the untrained COCO model already see?

If your objects are among the 80 COCO classes, this shows what you need to beat.

In [ ]:
# Load the pretrained COCO model without any custom training
baselineModel = YOLO(baseModelName)
# Predict on the validation images and save annotated copies to a fixed folder
baselineResults = baselineModel.predict(source=str(datasetRootPath / "images" / "val"), save=True, conf=0.4, project="/content/runs/baseline", name="predict", exist_ok=True)
# Show the first few annotated images
showSavedImages("/content/runs/baseline/predict")

## 7. Train

Results are written straight to Drive so the weights survive a disconnect. The first epochs can look slow, which is normal.

In [ ]:
# Load the pretrained model and fine-tune it on the custom dataset
model = YOLO(baseModelName)
# The project and name arguments fix the output folder so that later cells can find the weights
trainingResults = model.train(data=str(datasetYamlPath), epochs=trainingEpochs, imgsz=imageSize, project=str(driveResultsFolder / "train"), name="occlusion", exist_ok=True)

### If Colab disconnected during training

Uncomment the lines below, run the earlier setup cells (1 to 5), then run this cell to continue from the last checkpoint.

In [ ]:
# Reload the last checkpoint that was saved on Drive and carry on training from there
# resumeCheckpointPath = driveResultsFolder / "train" / "occlusion" / "weights" / "last.pt"
# resumeModel = YOLO(str(resumeCheckpointPath))
# resumeResults = resumeModel.train(resume=True)

## 8. Evaluate the trained model

In [ ]:
# Path to the best weights chosen automatically by validation score
bestWeightsPath = driveResultsFolder / "train" / "occlusion" / "weights" / "best.pt"
trainedModel = YOLO(str(bestWeightsPath))
# Run validation on the validation split and print the headline scores
validationMetrics = trainedModel.val(data=str(datasetYamlPath))
print(f"mAP50: {validationMetrics.box.map50:.3f}")
print(f"mAP50-95: {validationMetrics.box.map:.3f}")
# Show the training curves that Ultralytics saved as an image
display(Image.open(driveResultsFolder / "train" / "occlusion" / "results.png"))

## 9. Look at real predictions

Scores hide failures, so look at the images, especially the heavily occluded ones.

In [ ]:
# Predict on the validation images with the trained model and save annotated copies
predictionResults = trainedModel.predict(source=str(datasetRootPath / "images" / "val"), save=True, conf=0.4, project="/content/runs/trained", name="predict", exist_ok=True)
showSavedImages("/content/runs/trained/predict", maximumCount=8)

## 10. Evaluate by occlusion level (optional)

This is the cell that turns the project into a finding: how fast does accuracy fall as objects get more hidden?

In [ ]:
# Score each occlusion level separately and print the results side by side
occlusionScores = evaluateByOcclusionLevel(trainedModel, datasetRootPath, classNames, occlusionLevels)
for levelName, scoreValue in occlusionScores.items():
    print(f"{levelName}: mAP50 = {scoreValue:.3f}")

## 11. Export and download (optional)

Download `best.pt` for the local webcam script. Export only if you need another format.

In [ ]:
from google.colab import files
# Download the best weights to your own computer
files.download(str(bestWeightsPath))
# Export to ONNX; change the format to "tflite" or "coreml" if you need those instead
exportedModelPath = trainedModel.export(format="onnx")
print(exportedModelPath)